In [2]:
!pip install pandas numpy

In [3]:
import pandas as pd

#### Load CSV


In [ ]:
df = pd.read_csv("papers.csv")

# Quick overview
print(df.shape)
print(df.columns)
df.head()

(11994, 10)
Index(['paperId', 'title', 'url', 'abstract', 'publicationVenue', 'venue',
       'publicationTypes', 'publicationDate', 'openAccessPdf', 'authors'],
      dtype='object')


,paperId,title,url,abstract,publicationVenue,venue,publicationTypes,publicationDate,openAccessPdf,authors
0,00ade9595195aacdfb809e34a1970f3d957c67fc,Towards Effective Bug Reproduction for Mobile ...,https://www.semanticscholar.org/paper/00ade959...,Bug reproduction is a critical task in softwar...,"{'id': 'e5756ae3-5e70-4cef-8391-fd415080c43e',...",International Conferences on Dependable System...,"JournalArticle, Conference",10-08-23,NaN,"Xin Li, Shengcheng Yu, Lifan Sun, Yue-Yue Liu,..."
1,030311bce40218629292fc09996a02e4c93236fc,Analysis and Design of Intelligent Music Playe...,https://www.semanticscholar.org/paper/030311bc...,Gesture recognition based on computer vision i...,NaN,2022 IEEE 8th International Conference on Comp...,Conference,09-12-22,NaN,"Yiliang Xing, Huajun Lei"
2,05e5d098ad58cd33c1128a2dee156d18b1163789,Realization of a Hybrid Face Detecting and Ver...,https://www.semanticscholar.org/paper/05e5d098...,Face recognition is a popular subject in compu...,NaN,NaN,[],17-01-18,https://doi.org/10.5120/ijca2018916133,"Mahmut Dirik, Davut Hanbay, A. F. Kocamaz"
3,06569c60731b39744bfd6fb78e1c6b13e7d302f1,I Don't Have That Much Data! Reusing User Beha...,https://www.semanticscholar.org/paper/06569c60...,NaN,"{'id': '97674398-f65b-4f07-9db4-590d762f126b',...",International Conference on Web Engineering,JournalArticle,09-06-20,NaN,"M. Bakaev, Maximilian Speicher, Sebastian Heil..."
4,08c53cfae1976d4c461cad021b7c77f61d79fcf3,Machine Learning-Based Prototyping of Graphica...,https://www.semanticscholar.org/paper/08c53cfa...,It is common practice for developers of user-f...,"{'id': 'c99cfe66-b71c-4ca4-bedd-26267b9cb068',...",IEEE Transactions on Software Engineering,JournalArticle,07-02-18,https://doi.org/10.1109/tse.2018.2844788,"Kevin Moran, Carlos Bernal-Cárdenas, Michael C..."


#### Drop Duplicates


In [5]:
df = df.drop_duplicates()

#### Drop Duplicates based on ID


In [6]:
df = df.drop_duplicates(subset=["paperId"])

#### Drop Duplicates based on title after cleaning


In [7]:
df["title_clean"] = df["title"].str.lower().str.strip()

df = df.drop_duplicates(subset=["title_clean"])

#### Drop Duplicates based on url


In [8]:
df = df.drop_duplicates(subset=["url"])


#### Keep papers published in the last 10 years


In [9]:
df["publicationDate"] = pd.to_datetime(df["publicationDate"], errors="coerce")

df = df[df["publicationDate"].dt.year >= 2016]


C:\Users\Al Shahbaz\AppData\Local\Temp\ipykernel_12828\1099806697.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["publicationDate"] = pd.to_datetime(df["publicationDate"], errors="coerce")


In [10]:
cv_keywords = [
    "computer vision",
    "deep learning",
    "cnn",
    "image",
    "vision",
    "object detection",
    "ocr",
    "visual",
    "screenshot"
]

se_keywords = [
    "software engineering",
    "software testing",
    "bug",
    "program analysis",
    "software maintenance",
    "gui",
    "user interface",
    "code",
    "software development",
    "sdlc"
]

In [11]:
def contains_keywords(text, keywords):
    if pd.isna(text):
        return False
    text = text.lower()
    return any(keyword in text for keyword in keywords)

# Combine title + abstract for better filtering
df["combined_text"] = df["title"].fillna('') + " " + df["abstract"].fillna('')

df["cv_match"] = df["combined_text"].apply(lambda x: contains_keywords(x, cv_keywords))
df["se_match"] = df["combined_text"].apply(lambda x: contains_keywords(x, se_keywords))

df = df[(df["cv_match"]) & (df["se_match"])]

print("After CV + SE filtering:", len(df))

After CV + SE filtering: 2881


In [12]:
irrelevant_keywords = [
    "medical",
    "disease",
    "agriculture",
    "retinal",
    "drone",
    "face recognition",
    "gesture recognition",
    "human activity recognition"
]

def is_irrelevant(text):
    if pd.isna(text):
        return False
    text = text.lower()
    return any(keyword in text for keyword in irrelevant_keywords)

df = df[~df["combined_text"].apply(is_irrelevant)]

print("After removing irrelevant domains:", len(df))

After removing irrelevant domains: 2486


In [13]:
df = df.drop(columns=["title_clean", "combined_text", "cv_match", "se_match"], errors="ignore")


In [14]:
output_file = "filtered_papers.csv"
df.to_csv(output_file, index=False)
